# CNC 밀링 머신 공구 마모 탐지 — 실습 (클린 버전)

> **세션 1 · 제조 데이터 심층 분석**  |  데이터: *CNC Mill Tool Wear* (University of Michigan SMART Lab)

---

## 0. 이 노트북의 목표

> **"센서 데이터만 보고, 이 공구가 닳았는지(worn) 멀쩡한지(unworn) 알 수 있을까?"**

공구가 마모된 채 가공하면 제품 불량·가공 실패·설비 손상으로 이어집니다. 그래서 공구 마모 탐지는
**예지 보전(Predictive Maintenance)** 의 출발점입니다.

이 버전은 **처음부터 데이터 누수 없이, 올바른 평가 방법(실험 단위 분할)** 으로 진행합니다.
EDA → 특징공학 → 올바른 분할(GroupKFold) → 모델링·평가 → **Claude Code 실습** 순서입니다.

### 산출물 스펙 매핑
| 스펙 항목 | 다루는 섹션 |
|---|---|
| 1-1 핵심 알고리즘 | 4(Wavelet 특징) · 5(LightGBM·SMOTE) |
| 1-2 EDA(데이터 특성) | 3. EDA |
| 1-3 왜 이렇게 푸는가 | 5. 실험 단위 분할·유효표본 |
| 1-4 코드별 설명 | 전 코드 셀 하단 [코드 설명]/[해석] |

---
## 1. 도메인 이해 — CNC 밀링과 공구 마모

### 1.1 CNC 밀링 머신
컴퓨터 프로그램(G-code)대로 회전 절삭 공구를 움직여 소재를 깎는 기계입니다.
- X / Y / Z 축: 직선 이동 / **S 축(주축)**: 공구를 회전시키는 축(절삭 에너지가 나오는 곳)

### 1.2 공구 마모가 왜 중요한가
절삭날이 닳으면 절삭 저항·마찰이 커집니다. 같은 명령에도 추종 오차가 커지고, 전류·전력 소비가 늘죠.
마모 시점을 놓치면 불량 양산 + 공구 파손 → 그래서 마모 탐지가 예지 보전의 첫 단추입니다.

### 1.3 데이터셋: UMich SMART Lab "CNC Mill Tool Wear"
미시간대 SMART 연구실의 **벤치마크 데이터셋**입니다. 실제 공장 데이터가 아니라, "센서로 공구 마모를
탐지하는 방법"을 안전·저렴하게 개발·검증하려고 **통제된 실험실에서 수집한** 데이터예요.

- **왜 왁스(양초 재질) 블록?** 같은 실험을 18번 반복하려면 금속은 비싸고 느립니다. 왁스는 빠르고 안전하게
  반복할 수 있고, 목적이 "센서 신호 수집"이라 소재 자체는 중요하지 않습니다.
- **'S'자를 깎았다?** 공구가 지나가는 가공 경로(모양)가 S자입니다(연구실 SMART의 S). 직선·곡선·방향전환이
  섞이도록 설계된 테스트 형상이에요.
- **구성**: 이송속도·고정압력·공구상태를 조합해 18회, 100ms(10Hz)로 48개 센서값 기록.
  - experiment_01~18.csv: 48컬럼 시계열 / train.csv: 실험 단위 메타 + 라벨(tool_condition 등)

> **참고 — 이 데이터는 '진동(가속도계) 신호'가 아니다**
> 48컬럼은 가속도계 고주파 진동이 아니라 CNC 컨트롤러가 10Hz로 기록한 제어 텔레메트리입니다.
> 나이퀴스트가 5Hz라 **채터링(수백 Hz~kHz)** 같은 진동 시그니처는 애초에 담기지 않습니다.
> (채터링을 직접 다루는 실습은 6절에서 '합성 신호'로 따로 해봅니다.)

---
## 2. 환경 설정 & 데이터 로드  ` [1-4 코드별 설명]`

> **실행 환경** — Colab / 로컬 모두 지원. 첫 셀이 환경을 감지해 (Colab이면) 패키지·한글폰트를 설치합니다.

In [1]:
# --- [환경 설정] Colab 자동 감지 ---
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run("pip install -q PyWavelets imbalanced-learn lightgbm", shell=True)
    subprocess.run("apt-get -qq -y install fonts-nanum > /dev/null 2>&1", shell=True)
    import matplotlib.font_manager as fm
    try: fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    except Exception: pass
    print("Colab 환경 설정 완료")
else:
    print("로컬 환경")

In [2]:
import os, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import pywt
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid"); plt.rcParams["axes.unicode_minus"] = False
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)

import matplotlib.font_manager as _fm
_avail = {f.name for f in _fm.fontManager.ttflist}
for _f in ["AppleGothic", "NanumGothic", "Malgun Gothic"]:
    if _f in _avail:
        plt.rcParams["font.family"] = _f; print("한글 폰트:", _f); break
else:
    print("한글 폰트 미발견")

### 데이터 준비
로컬은 `data/` 폴더에 CSV가 있으면 됩니다. Colab이면 `data.zip`을 업로드한 뒤 한 줄로 풉니다.

```python
!unzip data.zip -d data/
```

In [3]:
def find_data_dir():
    for base in ["data", ".", "/kaggle/input/tool-wear-detection-in-cnc-mill"]:
        hits = glob.glob(os.path.join(base, "**", "train.csv"), recursive=True)
        if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError("train.csv 없음 → data/ 폴더에 데이터셋을 넣어주세요.")
DATA_DIR = find_data_dir(); print("데이터 경로:", DATA_DIR)
exp_meta = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
exp_meta["passed_visual_inspection"] = exp_meta["passed_visual_inspection"].fillna("no")
exp_meta[["No","feedrate","clamp_pressure","tool_condition"]]

### 18개 파일 통합 + 실험번호 남기기
센서 측정값 18개 파일에 각 실험의 정답(tool_condition)을 붙여 한 표로 만듭니다.
이때 **실험 번호(exp_num)** 를 꼭 남깁니다 — 5절에서 "같은 실험은 한쪽에만" 강제할 때 그룹 키로 씁니다.

> ⚠️ machining_finalized·passed_visual_inspection은 **가공이 끝난 뒤** 알 수 있는 값이라, 입력 피처로 쓰면
> "정답 컨닝"(타깃 누수)이 됩니다. 그래서 처음부터 피처에 넣지 않습니다.

In [4]:
frames = []
for i in range(1, 19):
    ts = pd.read_csv(os.path.join(DATA_DIR, f"experiment_{i:02d}.csv"))
    row = exp_meta[exp_meta["No"] == i].iloc[0]
    ts["exp_num"] = i
    ts["target"] = int(row["tool_condition"] == "worn")   # worn=1, unworn=0
    frames.append(ts)
df = pd.concat(frames, ignore_index=True)
df["Machining_Process"] = df["Machining_Process"].replace({"Starting": "Prep", "end": "End"})
print("통합:", df.shape)